In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import tensorflow as tf

In [3]:
df = pd.read_csv("../data/TSLA.csv")

In [4]:
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,2010-06-29,3.800,5.000,3.508,4.778,4.778,93831500
1,2010-06-30,5.158,6.084,4.660,4.766,4.766,85935500
2,2010-07-01,5.000,5.184,4.054,4.392,4.392,41094000
3,2010-07-02,4.600,4.620,3.742,3.840,3.840,25699000
4,2010-07-06,4.000,4.000,3.166,3.222,3.222,34334500


In [5]:
df.describe()

,Open,High,Low,Close,Adj Close,Volume
count,2813.000000,2813.000000,2813.000000,2813.000000,2813.000000,2.813000e+03
mean,98.307804,100.386953,96.069256,98.349877,98.349877,3.160441e+07
std,176.108180,179.847860,171.937613,176.160413,176.160413,2.857384e+07
min,3.228000,3.326000,2.996000,3.160000,3.160000,5.925000e+05
25%,8.838000,9.228000,8.782000,9.090000,9.090000,1.232800e+07
50%,45.720001,46.307999,44.908001,45.664001,45.664001,2.513300e+07
75%,63.998001,65.334000,63.014000,64.269997,64.269997,4.050650e+07
max,891.380005,900.400024,871.599976,883.090027,883.090027,3.046940e+08


In [6]:
df.isnull().sum()

Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2813 entries, 0 to 2812
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       2813 non-null   str    
 1   Open       2813 non-null   float64
 2   High       2813 non-null   float64
 3   Low        2813 non-null   float64
 4   Close      2813 non-null   float64
 5   Adj Close  2813 non-null   float64
 6   Volume     2813 non-null   int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 154.0 KB


In [8]:
df["ret"] = np.log(df["Close"]).diff()                 
df["vol_chg"] = np.log(df["Volume"]).diff()            
df["hl_range"] = (df["High"] - df["Low"]) / df["Close"]
df = df.dropna().reset_index(drop=True)

In [9]:
feature_cols = ["ret", "vol_chg", "hl_range"]

In [10]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

In [11]:
X_raw = df[feature_cols].values
y_raw = df["ret"].values

In [13]:
from sklearn.preprocessing import StandardScaler

In [14]:
scaler_X = StandardScaler().fit(X_raw[:train_end])
scaler_y = StandardScaler().fit(y_raw[:train_end].reshape(-1, 1))
Xs = scaler_X.transform(X_raw)
ys = scaler_y.transform(y_raw.reshape(-1, 1)).ravel()

In [15]:
L = 20

In [16]:
tf.random.set_seed(42)
np.random.seed(42)

In [17]:
def make_windows(X, y, L):
    Xw, yw, days = [], [], []
    for d in range(L, len(X)):
        Xw.append(X[d - L:d])       
        yw.append(y[d])             
        days.append(d)
    return np.array(Xw), np.array(yw), np.array(days)

In [18]:
Xw, yw, days = make_windows(Xs, ys, L)


In [19]:
tr = days < train_end
va = (days >= train_end) & (days < val_end)
te = days >= val_end
print(f"Train: {tr.sum()}  Val: {va.sum()}  Test: {te.sum()}")

Train: 1948  Val: 422  Test: 422


In [20]:
from tensorflow import keras

In [21]:
model = keras.Sequential([
    keras.layers.Input(shape=(L, len(feature_cols))),
    keras.layers.LSTM(32),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

In [22]:
early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
model.fit(
    Xw[tr], yw[tr],
    validation_data=(Xw[va], yw[va]),
    epochs=100, batch_size=32,
    shuffle=True,       # OK: each window is self-contained; the split is what matters
    callbacks=[early], verbose=0,
)

In [23]:
actual = y_raw[days[te]]


In [24]:
p_lstm = scaler_y.inverse_transform(model.predict(Xw[te], verbose=0)).ravel()


In [25]:
from sklearn.linear_model import Ridge


In [26]:
ridge = Ridge(alpha=1.0).fit(Xw[tr].reshape(tr.sum(), -1), yw[tr])
p_ridge = scaler_y.inverse_transform(
    ridge.predict(Xw[te].reshape(te.sum(), -1)).reshape(-1, 1)
).ravel()

In [27]:
from sklearn.metrics import mean_squared_error


In [28]:
train_mean = y_raw[days[tr]].mean()


In [29]:
p_mean = np.full_like(actual, train_mean)


In [30]:
def report(name, p):
    mse = mean_squared_error(actual, p)
    dir_acc = ((p > 0) == (actual > 0)).mean()
    print(f"{name:14s}  MSE={mse:.6f}   DirAcc={dir_acc:.3f}")


In [31]:
print(f"Test days: {len(actual)}  |  Fraction of up days: {(actual > 0).mean():.3f}")
report("Mean baseline", p_mean)
report("Ridge", p_ridge)
report("LSTM", p_lstm)

Test days: 422  |  Fraction of up days: 0.545
Mean baseline   MSE=0.002397   DirAcc=0.545
Ridge           MSE=0.002510   DirAcc=0.524
LSTM            MSE=0.002404   DirAcc=0.545
